# Gemma 4 via Ollama Cloud API

Run every notebook in this series **without your desktop Ollama running** by
pointing at Ollama's hosted cloud endpoint instead.

## How to get your cloud API key
1. Open the **Ollama app** on your phone
2. Tap **Cloud API access → Create API key**
3. Copy the key and set it as an env var:

```bash
export OLLAMA_CLOUD_API_KEY=sk-ollama-...
```

That's it — no GPU, no local server, just your phone key.

## Cloud vs local model names

Ollama Cloud hosts a different model catalogue than the local Ollama server:

| Mode | Gemma 4 model ID | Notes |
|---|---|---|
| **Cloud** | `gemma4:31b` | Only Gemma 4 size available on cloud |
| **Local** | `gemma4:12b` | Default local pull (also `gemma4:27b`) |

`get_gemma_llm()` auto-picks the right default for whichever mode is active.

## Architecture
- **Local Ollama** (`http://localhost:11434`): uses `llama_index.llms.ollama.Ollama` directly
- **Cloud Ollama** (`https://ollama.com/v1`): uses `OpenAILike` with OpenAI-compatible endpoint
- The `get_gemma_llm()` helper below auto-detects which to use based on env vars

In [ ]:
import os
from typing import Union

# ---------------------------------------------------------------------------
# Environment variables (set these before running any notebook in this series)
# ---------------------------------------------------------------------------
# Local Ollama (default):  no env vars needed if Ollama is running on localhost
# Cloud Ollama:            set OLLAMA_CLOUD_API_KEY (from the Ollama app)
# Custom endpoint:         set OLLAMA_BASE_URL to override the default
# Model override:          set OLLAMA_MODEL (cloud default: gemma4:31b, local default: gemma4:12b)

OLLAMA_CLOUD_API_KEY = os.environ.get("OLLAMA_CLOUD_API_KEY", "")
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "")  # optional override
REQUEST_TIMEOUT = float(os.environ.get("OLLAMA_TIMEOUT", "180"))

In [ ]:
def get_gemma_llm(model: str = "", timeout: float = REQUEST_TIMEOUT):
    """
    Returns a LlamaIndex LLM connected to Gemma 4, auto-selecting:
      - Ollama Cloud (via OpenAILike) when OLLAMA_CLOUD_API_KEY is set
      - Local Ollama when running on localhost

    Cloud default model:  gemma4:31b  (the Gemma 4 build available on Ollama Cloud)
    Local default model:  gemma4:12b  (standard local pull)
    Override either with:  OLLAMA_MODEL=<name>
    """
    if OLLAMA_CLOUD_API_KEY:
        # Ollama Cloud — OpenAI-compatible endpoint
        try:
            from llama_index.llms.openai_like import OpenAILike
        except ImportError:
            raise ImportError(
                "Install llama-index-llms-openai-like for cloud mode:\n"
                "  pip install llama-index-llms-openai-like"
            )
        _model = model or os.environ.get("OLLAMA_MODEL", "gemma4:31b")
        base_url = OLLAMA_BASE_URL or "https://ollama.com/v1"
        print(f"Using Ollama CLOUD at {base_url} — model: {_model}")
        return OpenAILike(
            model=_model,
            api_base=base_url,
            api_key=OLLAMA_CLOUD_API_KEY,
            is_chat_model=True,
            is_function_calling_model=True,
            context_window=128_000,
            timeout=timeout,
        )
    else:
        # Local Ollama
        from llama_index.llms.ollama import Ollama
        _model = model or os.environ.get("OLLAMA_MODEL", "gemma4:12b")
        base_url = OLLAMA_BASE_URL or "http://localhost:11434"
        print(f"Using LOCAL Ollama at {base_url} — model: {_model}")
        return Ollama(
            model=_model,
            base_url=base_url,
            request_timeout=timeout,
        )


llm = get_gemma_llm()
print(f"LLM ready: {type(llm).__name__}")

In [ ]:
# Quick smoke test
from llama_index.core.llms import ChatMessage

response = await llm.achat([
    ChatMessage(role="user", content="Say 'Ollama Cloud is working' if you can read this.")
])
print(response.message.content)

## Drop-in replacement for all other notebooks

In any other notebook in this series, replace the first cell that has:
```python
from llama_index.llms.ollama import Ollama
llm = Ollama(model="gemma4:12b", request_timeout=120.0)
```

with:
```python
%run gemma4_ollama_cloud.ipynb   # or copy the get_gemma_llm() function
llm = get_gemma_llm()
```

Everything else stays the same — agents, RAG, vision, MCP tools.

## Updated FastAPI server — supports both local and cloud

This replaces the `gemma4_server.py` generated in `gemma4_vercel_chat.ipynb`.
Run it on your desktop (or any server) and it automatically uses cloud Ollama
if `OLLAMA_CLOUD_API_KEY` is set.

In [ ]:
FASTAPI_SERVER_CODE = '''
from __future__ import annotations

import asyncio
import json
import os
from typing import AsyncGenerator

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

# ── LLM selection ──────────────────────────────────────────────────────────
CLOUD_API_KEY = os.environ.get("OLLAMA_CLOUD_API_KEY", "")
BASE_URL = os.environ.get("OLLAMA_BASE_URL", "")
# Cloud has gemma4:31b; local has gemma4:12b — pick the right default
_default_model = "gemma4:31b" if CLOUD_API_KEY else "gemma4:12b"
MODEL = os.environ.get("OLLAMA_MODEL", _default_model)

if CLOUD_API_KEY:
    from llama_index.llms.openai_like import OpenAILike
    llm = OpenAILike(
        model=MODEL,
        api_base=BASE_URL or "https://ollama.com/v1",
        api_key=CLOUD_API_KEY,
        is_chat_model=True,
        context_window=128_000,
        timeout=180.0,
    )
    print(f"[server] Using Ollama CLOUD — model: {MODEL}")
else:
    from llama_index.llms.ollama import Ollama
    llm = Ollama(
        model=MODEL,
        base_url=BASE_URL or "http://localhost:11434",
        request_timeout=180.0,
    )
    print(f"[server] Using LOCAL Ollama — model: {MODEL}")

from llama_index.core.llms import ChatMessage

# ── App ────────────────────────────────────────────────────────────────────
app = FastAPI(title="Gemma 4 Chat API")

ALLOWED_ORIGINS = os.environ.get("ALLOWED_ORIGINS", "http://localhost:3000").split(",")
app.add_middleware(
    CORSMiddleware,
    allow_origins=ALLOWED_ORIGINS,
    allow_methods=["POST", "GET"],
    allow_headers=["*"],
)


class ChatRequest(BaseModel):
    messages: list[dict]


@app.get("/health")
async def health() -> dict:
    mode = "cloud" if CLOUD_API_KEY else "local"
    return {"status": "ok", "model": MODEL, "mode": mode}


@app.post("/chat")
async def chat(req: ChatRequest) -> dict:
    messages = [ChatMessage(role=m["role"], content=m["content"]) for m in req.messages]
    response = await llm.achat(messages)
    return {"content": str(response.message.content)}


async def _stream_tokens(messages: list[ChatMessage]) -> AsyncGenerator[str, None]:
    async for chunk in await llm.astream_chat(messages):
        if chunk.delta:
            yield f"data: {json.dumps({\x27delta\x27: chunk.delta})}\\n\\n"
    yield "data: [DONE]\\n\\n"


@app.post("/stream")
async def stream_chat(req: ChatRequest) -> StreamingResponse:
    messages = [ChatMessage(role=m["role"], content=m["content"]) for m in req.messages]
    return StreamingResponse(
        _stream_tokens(messages),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
    )
'''

with open("gemma4_server.py", "w") as f:
    # Fix the escaped quote used to avoid f-string nesting issues
    f.write(FASTAPI_SERVER_CODE.strip().replace("\\x27", "'"))

print("gemma4_server.py written.")
print()
print("Local:  uvicorn gemma4_server:app --host 0.0.0.0 --port 8080")
print("Cloud:  OLLAMA_CLOUD_API_KEY=sk-... uvicorn gemma4_server:app --host 0.0.0.0 --port 8080")

## Phone-only workflow (no desktop needed)

With Ollama Cloud:

```
Phone (browser) → Vercel frontend → any server (or localhost) → Ollama Cloud → gemma4:31b
```

You can even run the FastAPI server on a free cloud VM (Railway, Render, fly.io)
with just `OLLAMA_CLOUD_API_KEY` set — no GPU needed, no desktop.

```bash
# Deploy to Railway (free tier)
# 1. Push gemma4_server.py to a repo
# 2. railway new → connect repo → add env var OLLAMA_CLOUD_API_KEY
# 3. Update Vercel NEXT_PUBLIC_BACKEND_URL to your Railway URL
# 4. Open Vercel URL on phone — fully serverless Gemma 4 chat
```

## Vision (image input) on cloud

The `OpenAILike` backend supports image input if the cloud endpoint accepts it.
Use the same `ImageBlock` + `TextBlock` pattern — it maps to OpenAI's `image_url`
content type automatically via LlamaIndex.

In [ ]:
# Vision test (cloud or local)
import base64
from pathlib import Path
from llama_index.core.base.llms.types import ImageBlock, TextBlock

IMAGE_PATH = os.environ.get("TEST_IMAGE", "")

if IMAGE_PATH and Path(IMAGE_PATH).exists():
    with open(IMAGE_PATH, "rb") as f:
        image_data = f.read()
    from llama_index.core.llms import ChatMessage
    response = await llm.achat([
        ChatMessage(
            role="user",
            blocks=[
                ImageBlock(image=image_data),
                TextBlock(text="Describe this image in one sentence."),
            ],
        )
    ])
    print(response.message.content)
else:
    print("Set TEST_IMAGE to an image path to test vision on cloud.")

## Environment variable cheatsheet

| Variable | Default | Description |
|---|---|---|
| `OLLAMA_CLOUD_API_KEY` | _(unset = local)_ | API key from the Ollama app → enables cloud mode |
| `OLLAMA_BASE_URL` | auto | Override endpoint URL (cloud or custom self-hosted) |
| `OLLAMA_MODEL` | `gemma4:31b` (cloud) / `gemma4:12b` (local) | Model to use |
| `OLLAMA_TIMEOUT` | `180` | Request timeout in seconds |
| `ALLOWED_ORIGINS` | `http://localhost:3000` | CORS origins for the FastAPI server |

### Available cloud models (as of June 2026)
Models confirmed available on `https://ollama.com/v1`:
- **`gemma4:31b`** — Gemma 4 (the only Gemma 4 size on cloud)
- `gemma3:4b`, `gemma3:12b`, `gemma3:27b` — Gemma 3 family
- `deepseek-v3.2`, `qwen3-coder:480b`, `kimi-k2.7-code`, and many others

To list all available models for your key:
```bash
curl https://ollama.com/v1/models -H "Authorization: Bearer $OLLAMA_CLOUD_API_KEY" | python3 -m json.tool
```